In [1]:
#The following file paths are all absolute paths. You can replace them with relative paths at runtime, and the files are located in their respective folders.
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
import matplotlib.pyplot as plt
import random
import argparse
import time
from collections import OrderedDict
from copy import copy
# import Learn_Knonlinear as lka
import scipy
import scipy.linalg
from scipy.integrate import odeint
from tqdm import tqdm, trange
import sys
import os
import csv
# os.chdir(r'/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
os.chdir(r'D:/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
sys.path.append("utility_LSPN/")
sys.path.append("LSPN_compare/sizeNN_learnmodel_train")
sys.path.append("utility_LSPN/")
try:
    from LSPN_test import LSPN_Mamba
except:
    pass
from Utility import data_collecter
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"

In [2]:
Methods = ["KNonlinear","KNonlinearRNN","KoopmanU",\
            "KoopmanNonlinearA","KoopmanNonlinear",\
                "KNonlinearmamba"]
Method_names = ["KDNN","KRNN","DKUC(no SOC)",\
            "DKAC(no SOC)","DKN(no SOC)","KNonlinearmamba"]

In [3]:
def read_lorenz_dataset_original_shape(file_path, num_samples=100, num_steps=100):
    data = np.zeros((num_samples, num_steps, 3))
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # 跳过列名行
        for i in range(num_samples):
            for j in range(num_steps):
                row = next(reader)
                data[i, j] = [float(val) for val in row]
    return data
def eval_err(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "DATA/LSPN_compare_sizeNNdata/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method_index==1:
        import Learn_Knonlinear_RNN_luorenz as lka
    elif method_index==0:
        import Learn_Knonlinear_luorenz as lka
    elif method_index==2:
        import Learn_DKUC_withoutSOC_luorenz as lka
    elif method_index==3:
        import Learn_DKAC_withoutSOC_luorenz as lka
    elif method_index==4:
        import Learn_DKN_withoutSOC_luorenz as lka
    elif method_index==5:
        import Learn_Knonlinear_mamba_luorenz as lka
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+env_name+"layer{}".format(layer_i)+"_") and file.endswith(".pth"):
            model_path = file  
    # Data_collect = data_collecter(env_name)
    udim = 0
    Nstates = 3
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=udim)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=udim+Nstates,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim) 
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda") 
    net.load_state_dict(state_dict)
    total_params = sum(p.numel() for p in net.parameters())
    print(f"{method} Total parameters: {total_params}")
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 50
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 10
    max_loss_all = np.zeros((times,steps-1))
    mean_loss_all = np.zeros((times,steps-1))
    min_loss_all = np.zeros((times,steps-1))
    Test_time = np.zeros((times))
    test_loss = []
    test_loss_log = []
    with torch.no_grad():
        for i in trange(times, desc="predicting", unit="times"):
            start_time = time.time()
            # test_data_path = "DATA/LSPN_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            X_original_shape = read_lorenz_dataset_original_shape('/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/lorenz_data100000.csv',num_samples=Samples,num_steps=steps)
            test_data = X_original_shape[-Samples:, :steps, :]
            np.save("DATA/LSPN_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i),test_data)
            max_loss,mean_loss,min_loss = lka.K_loss(test_data,net,udim,Nstate=Nstates)
            max_loss_all[i] = max_loss.reshape(-1)
            mean_loss_all[i] = mean_loss.reshape(-1)
            test_loss.append(mean_loss.reshape(-1))
            test_loss_log.append(np.log10(mean_loss_all[i]))
            min_loss_all[i] = min_loss.reshape(-1)
            end_time = time.time()
            t_cost = end_time - start_time
            Test_time[i] = t_cost
            print(Test_time[i])
    max_mean = np.mean(max_loss_all,axis=0)
    max_std = np.std(max_loss_all,axis=0)
    mean_mean =  np.mean(mean_loss_all,axis=0)
    mean_std =  np.std(mean_loss_all,axis=0)
    min_mean =  np.mean(min_loss_all,axis=0)
    min_std =  np.std(min_loss_all,axis=0)  
    test_loss = np.array(test_loss)
    test_loss_log = np.array(test_loss_log)
    print("test_loss:{}".format(test_loss_log.shape))
    print("test_time:{}".format(Test_time.shape))
    # np.save("DATA/LSPN_compare_drawdata/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([max_mean,max_std,mean_mean,mean_std,min_mean,min_std]))
    np.save("DATA/LSPN_compare_drawdata/log_"+env_name+"{}".format(method)+"_{}.npy".format(steps),test_loss_log)
    np.save("DATA/LSPN_compare_drawdata/"+env_name+"{}".format(method)+"_{}.npy".format(steps),test_loss)
    np.save("DATA/LSPN_compare_drawdata/time_"+env_name+"{}".format(method)+"_{}.npy".format(steps),Test_time)
    return max_mean,max_std,mean_mean,mean_std,min_mean,min_std

In [4]:
suffix = ["Knolinear_SOC_models_luorenz","KRNN_SOC_models_luorenz","DKUC_withoutSOC_sizeNN_luorenz","DKAC_withoutSOC_sizeNN_luorenz","DKN_withoutSOC_sizeNN_luorenz","mamba_testluorenz2"]
env_name = "luorenz"

steps = 10001
for i in [5]:#0,1,2,3,4,5
    eval_err(suffix[i],env_name,method_index=i,layer_i=4, steps = steps)

KNonlinearmamba
KNonlinearmamba Total parameters: 504


predicting:  10%|█         | 1/10 [00:01<00:15,  1.69s/times]

1.690413475036621


predicting:  20%|██        | 2/10 [00:03<00:13,  1.63s/times]

1.585686445236206


...

1.630000352859497


predicting:  90%|█████████ | 9/10 [00:14<00:01,  1.59s/times]

1.562638759613037


predicting: 100%|██████████| 10/10 [00:15<00:00,  1.59s/times]

1.5551600456237793
test_loss:(10, 10000)
test_time:(10,)


In [6]:
for k in [0,1,2,3,4,5]:#0,1,2,3,4,5
    env_name = "luorenz"
    method = Methods[k]
    steps = 10001
    data_all = np.load("DATA/LSPN_compare_drawdata/"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    time_cost = np.load("DATA/LSPN_compare_drawdata/time_"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    Loss_Data = []
    mean_data = []
    for i in range(steps-1):#15
        data = data_all[:,i]
        # 计算平均值和标准差
        mean_value = sum(data) / len(data)
        std_deviation = (sum((x - mean_value) ** 2 for x in data) / len(data)) ** 0.5

        # 使用科学计数法格式化字符串
        formatted_string = "{:.3e} ± {:.3e}".format(mean_value, std_deviation)
        Loss_Data.append(formatted_string)
        # print(formatted_string)
        mean_data.append(mean_value)
    Loss_Data = np.array(Loss_Data)
    mean_data = np.array(mean_data)
    print(method)
    print(Loss_Data[14],Loss_Data[29],Loss_Data[49],Loss_Data[99])
    # np.save("DATA/LSPN_compare_drawdata/{}{}_{}step_loss.npy".format(env_name, method, steps),Loss_Data)

KNonlinear
1.943e+00 ± 4.441e-16 4.220e+00 ± 8.882e-16 7.734e+00 ± 0.000e+00 8.992e+00 ± 0.000e+00
KNonlinearRNN
1.522e-01 ± 0.000e+00 1.824e-01 ± 2.776e-17 1.024e-01 ± 0.000e+00 8.187e-02 ± 1.388e-17
KoopmanU
8.824e-01 ± 1.110e-16 8.248e+00 ± 1.776e-15 1.572e+01 ± 1.776e-15 1.132e+02 ± 0.000e+00
KoopmanNonlinearA
1.022e+00 ± 2.220e-16 9.533e+00 ± 1.776e-15 1.903e+01 ± 3.553e-15 1.942e+02 ± 0.000e+00


C:\Users\25843\AppData\Local\Temp\ipykernel_17880\3021886183.py:13: RuntimeWarning: overflow encountered in double_scalars
  std_deviation = (sum((x - mean_value) ** 2 for x in data) / len(data)) ** 0.5


KoopmanNonlinear
1.132e+00 ± 2.220e-16 8.052e+00 ± 1.776e-15 1.410e+01 ± 0.000e+00 5.402e+01 ± 7.105e-15
KNonlinearmamba
9.475e-03 ± 0.000e+00 8.858e-03 ± 0.000e+00 6.608e-03 ± 0.000e+00 9.386e-03 ± 0.000e+00
